<a href="https://colab.research.google.com/github/boyerdan2000-hipr/WarpX-PIC-Beam-Plasma-Wakefield-Propulsion/blob/main/hipr_picmi_dynamic_gradient_compressed_beam.py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1 (ONE-TIME SAVE): MONDAY CMAKE COMPILATION + TARBALL BACKUP
# ==============================================================================
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)
backup_dir = '/content/drive/My Drive/WarpX_L4_Compiled'
os.makedirs(backup_dir, exist_ok=True)

# 2. System dependencies
!apt-get update -y > /dev/null
!apt-get install -y cmake g++ libopenmpi-dev openmpi-bin libfftw3-dev > /dev/null
!pip install -q picmistandard periodictable

# 3. Clone fresh repository
!rm -rf /content/WarpX
!git clone https://github.com/ECP-WarpX/WarpX.git /content/WarpX
!mkdir -p /content/WarpX/build
%cd /content/WarpX/build

# 4. Native CMake Build (The proven Monday method)
print("\nCompiling WarpX natively with RZ Geometry (This will take ~15 mins)...")
!cmake -DWarpX_COMPUTE=CUDA -DWarpX_DIMS=RZ -DWarpX_PYTHON=ON ..
!make -j 4

# 5. Compress the entire working environment into a single file and save to Drive
print("\nCompressing build into a single archive...")
%cd /content
!tar -czf warpx_backup.tar.gz WarpX/
!cp warpx_backup.tar.gz "{backup_dir}/"

print("\n===================================================================")
print("   Compilation Complete & Tarball Saved to Google Drive!")
print("===================================================================")

In [ ]:
# ==============================================================================
# CELL 1 (FAST BOOT): EXTRACT TARBALL FROM GOOGLE DRIVE
# ==============================================================================
import os
import sys
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)
backup_file = '/content/drive/My Drive/WarpX_L4_Compiled/warpx_backup.tar.gz'

# 2. Lightweight dependencies
!apt-get update -y > /dev/null
!apt-get install -y libopenmpi-dev openmpi-bin libfftw3-dev > /dev/null
!pip install -q picmistandard periodictable

# 3. Extract the exact working environment
if os.path.exists(backup_file):
    print(f"Extracting WarpX environment from Drive...")
    !rm -rf /content/WarpX
    !cp "{backup_file}" /content/warpx_backup.tar.gz
    !tar -xzf /content/warpx_backup.tar.gz

    # Map Python paths directly to the extracted build (Monday's exact mapping)
    sys.path.insert(0, '/content/WarpX/build/lib')
    sys.path.insert(0, '/content/WarpX/build/bin')
    sys.path.insert(0, '/content/WarpX/Python')

    print("\n===================================================================")
    print("   WarpX Fast Boot Complete! Ready for Cell 2.")
    print("===================================================================")
else:
    print("Error: Backup archive not found.")

In [ ]:
# ==============================================================================
# CELL 2: WARPX PICMI INITIALIZATION (FIXED RZ PARSER & OPTIMIZED GRID)
# ==============================================================================
import os
import sys
import shutil
import glob
import numpy as np

# -------------------------------------------------------------------------
# 1. PATH HUNTER & BRUTE FORCE COPY
# -------------------------------------------------------------------------
warp_dir = '/content/WarpX'
sys.path.insert(0, os.path.join(warp_dir, 'Python'))
pywarpx_dir = '/content/WarpX/Python/pywarpx'

print("Hunting and placing compiled C++ modules...")
for root, dirs, files in os.walk(warp_dir):
    if 'amrex' in dirs and root not in sys.path:
        sys.path.insert(0, root)
        print(f"Found amrex module at: {root}")

    for f in files:
        if f.endswith('.so'):
            if root not in sys.path:
                sys.path.insert(0, root)
            if 'warpx_pybind' in f:
                try:
                    shutil.copy(os.path.join(root, f), pywarpx_dir)
                    print(f"Placed C++ binary: {f}")
                except shutil.SameFileError:
                    pass

# -------------------------------------------------------------------------
# 2. LOAD WARPX & DEFINE CONSTANTS
# -------------------------------------------------------------------------
# FIX: Import 'amr' to set AMReX-specific backend parameters globally
from pywarpx import picmi, amr
amr.max_grid_size = 2048
amr.blocking_factor = 32

c = 299792458.0
m_p = 1.67262192e-27
e_charge = 1.602176634e-19
n_plasma = 1.0e19

# 3. Domain & Grid setup
z_min, z_max, r_max = -30.0e-3, 2.0e-3, 1.5e-3
grid = picmi.CylindricalGrid(
    number_of_cells=[256, 2048],
    lower_bound=[0, z_min],
    upper_bound=[r_max, z_max],
    lower_boundary_conditions=['none', 'dirichlet'],
    upper_boundary_conditions=['dirichlet', 'dirichlet'],
    lower_boundary_conditions_particles=['none', 'absorbing'],
    upper_boundary_conditions_particles=['absorbing', 'absorbing'],
    n_azimuthal_modes=1
)
solver = picmi.ElectromagneticSolver(grid=grid, method='Yee', cfl=0.99)
plasma_layout = picmi.GriddedLayout(n_macroparticle_per_cell=[2, 2, 1], grid=grid)

# 4. Plasma Distributions
lithium_distribution = picmi.UniformDistribution(
    density=n_plasma,
    directed_velocity=[0.0, 0.0, 0.0]
)
electron_distribution = picmi.UniformDistribution(
    density=n_plasma,
    directed_velocity=[0.0, 0.0, 0.0]
)

electrons = picmi.Species(particle_type='electron', name='electrons', initial_distribution=electron_distribution)
lithium_ions = picmi.Species(particle_type='Li', charge_state=1, mass=7.016 * m_p, name='lithium_ions', initial_distribution=lithium_distribution)

# 5. Define Radially Compressed Gold Beam
v_b = 0.2 * c
gamma = 1.02062
u_z = v_b * gamma

beam_radius = 500e-6
packet_length = 1.055e-3
period = 2.11e-3
I_peak = 6.0
beam_area = np.pi * (beam_radius**2)
n0 = I_peak / (e_charge * beam_area * v_b)

k_p = 2.0 * np.pi / period
local_z = f"(-z - {period}*floor(-z/{period}))"
step_fraction = 0.15
doorstep_shape = f"({step_fraction} + (1.0 - {step_fraction}) * (exp({k_p} * {local_z}) - 1) / (exp({k_p} * {packet_length}) - 1))"

# FIX: Reverted back to 'x' for the string parser
doorstep_train = f"{n0} * (x < {beam_radius}) * (z > -21.1e-3) * (z <= 0.0) * " \
                 f"({local_z} < {packet_length}) * {doorstep_shape}"

gold_distribution = picmi.AnalyticDistribution(
    density_expression=doorstep_train,
    momentum_expressions=['0', '0', f"{u_z}"]
)
gold_beam = picmi.Species(particle_type='Au', charge_state=51, mass=196.97 * m_p, name='gold_beam', initial_distribution=gold_distribution)

# 6. Initialize Simulation
sim = picmi.Simulation(solver=solver, max_steps=120000, verbose=1)
sim.add_species(electrons, layout=plasma_layout)
sim.add_species(lithium_ions, layout=plasma_layout)
sim.add_species(gold_beam, layout=picmi.GriddedLayout(n_macroparticle_per_cell=[2, 4, 1], grid=grid))

print(f"\nCell 2 Complete: 0.2c Au51+ driver initialized in 1.0e19 m^-3 Li+ plasma.")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE
print("Forcing Google Drive connection...")
drive.mount('/content/drive')
save_dir = '/content/drive/My Drive/WarpX_Checkpoints'
os.makedirs(save_dir, exist_ok=True)

# 2. CHECKPOINT DISCOVERY (RESUME LOGIC)
print("Verifying Drive contents...")
existing_files = os.listdir(save_dir)
print(f"Files found in directory: {existing_files}")

chunk_files = [f for f in existing_files if f.startswith('ez_chunk_') and f.endswith('.npy')]
TOTAL_STEPS = 120000
CHUNK_SIZE = 4000
chunks_needed = TOTAL_STEPS // CHUNK_SIZE

if chunk_files:
    chunk_nums = [int(f.replace('ez_chunk_', '').replace('.npy', '')) for f in chunk_files]
    last_saved_step = max(chunk_nums)
    start_chunk = last_saved_step // CHUNK_SIZE
    print(f"\n[RESUME] Found {len(chunk_files)} chunks. Catching engine up to Step {last_saved_step}...")
    sim.step(last_saved_step)  # Silently catch up the physics engine
else:
    start_chunk = 0
    print("\n[NEW RUN] No chunks found. Starting from Step 0...")

# 3. MAIN PHYSICS LOOP
for chunk_idx in range(start_chunk + 1, chunks_needed + 1):
    print(f"\n--- Calculating Packet {chunk_idx} of {chunks_needed} ---")

    # 400-step heartbeat loop
    for hb in range(10):
        sim.step(400)
        current_step = (chunk_idx - 1) * CHUNK_SIZE + ((hb + 1) * 400)
        print(f"   Heartbeat: Step {current_step} complete...")

    # 4. EXTRACT AND SAVE
    print(f"Extracting fields at Step {chunk_idx * CHUNK_SIZE}...")

    # Safely grab the 1D slice directly from the GPU and pull it to CPU RAM
    # Using .copy() severs the connection to AMReX's C++ memory manager
    ez_1d_cpu = sim.fields.get("Efield_fp", dir="z", level=0)[0, :].copy()

    # Convert to MV/m and downsample
    ez_downsampled = (ez_1d_cpu / 1e6)[::10]

    # Save the chunk
    chunk_filename = os.path.join(save_dir, f'ez_chunk_{chunk_idx * CHUNK_SIZE}.npy')
    np.save(chunk_filename, ez_downsampled)

    # Calculate and save metrics
    stats = pd.DataFrame([{
        "chunk": chunk_idx,
        "max_ez": float(np.max(ez_1d_cpu) / 1e6),
        "min_ez": float(np.min(ez_1d_cpu) / 1e6),
        "avg_abs_ez": float(np.mean(np.abs(ez_1d_cpu)) / 1e6)
    }])
    stats.to_csv(os.path.join(save_dir, 'raw_stats.txt'), mode='a', header=False, index=False)

    print(f"-> Packet {chunk_idx} Saved to Google Drive.")

    # 5. GENTLE MEMORY PURGE (Standard Python GC only)
    del ez_1d_cpu, ez_downsampled, stats
    gc.collect()

In [ ]:
# ==============================================================================
# Cell 4: Data Analysis and Plotting (Au51+ / Li+ Baseline)
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
save_dir = '/content/drive/My Drive/WarpX_Checkpoints/'

print("Scanning Google Drive for saved checkpoints...")

# 2. Load exact raw AC statistics
stats_file = os.path.join(save_dir, 'raw_stats.txt')
if os.path.exists(stats_file):
    stats_df = pd.read_csv(stats_file)
    global_max_ez = stats_df['max_ez'].max()
    global_min_ez = stats_df['min_ez'].min()
    global_avg_abs_ez = stats_df['avg_abs_ez'].mean()
else:
    print("Warning: raw_stats.txt not found. Metrics will be skipped.")
    global_max_ez, global_min_ez, global_avg_abs_ez = 0, 0, 0

# 3. Locate, sort, and stitch the compressed array chunks
chunk_files = glob.glob(os.path.join(save_dir, 'ez_chunk_*.npy'))
chunk_files.sort(key=lambda x: int(x.split('_chunk_')[-1].split('.npy')[0]))

print(f"Successfully loaded {len(chunk_files)} snapshot chunks.")
compressed_snapshots = [np.load(f) for f in chunk_files]
final_snapshot = compressed_snapshots[-1]
final_step = int(chunk_files[-1].split('_chunk_')[-1].split('.npy')[0])

# 4. Reconstruct the spatial axis based on the FULL STATIONARY GRID
z_min, z_max = -30.0e-3, 2.0e-3
original_num_cells = 2048
original_dz = (z_max - z_min) / original_num_cells

stride = 10
new_dz = original_dz * stride
z_array_compressed = np.linspace(z_min, z_max, len(final_snapshot)) * 1e3 # mm

# 5. Apply Butterworth low-pass filter
fs = 1.0 / new_dz
nyquist = 0.5 * fs
cutoff = 0.1 * nyquist
b, a = butter(4, cutoff / nyquist, btype='low', analog=False)
Ez_dc_filtered = filtfilt(b, a, final_snapshot)


# ==============================================================================
# PLOT 1: Raw Unfiltered Wakefield Gradient with Target Lines
# ==============================================================================
plt.figure(figsize=(12, 5))
plt.plot(z_array_compressed, final_snapshot, label='Simulated $E_z$ Gradient', color='blue', linewidth=1)

# Target Lines
plt.axhline(100, color='red', linestyle='--', label='100 MV/m Peak Target')
plt.axhline(-100, color='red', linestyle='--')
plt.axhline(50, color='green', linestyle=':', label='50 MV/m Avg Target')
plt.axhline(-50, color='green', linestyle=':')
plt.axhline(0, color='black', linewidth=1)

plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.title('Longitudinal Wakefield Gradient ($E_z$)\n0.2c Au51+ Continuous Train', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# ==============================================================================
# PLOT 2: 1D Macroscopic Ambipolar Towing Field
# ==============================================================================
plt.figure(figsize=(12, 6))
plt.plot(z_array_compressed, final_snapshot, label='Downsampled Wakefield ($E_z$)', color='blue', alpha=0.25)
plt.plot(z_array_compressed, Ez_dc_filtered, label='Macroscopic DC Ambipolar Field', color='red', linewidth=3)
plt.axhline(0, color='black', linewidth=1)
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.title(f'Macroscopic Ponderomotive Acceleration & Ambipolar Towing\nLow-Pass Filtered Longitudinal Wakefield at Step {final_step:,}', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# ==============================================================================
# PLOT 3: 2D Spatiotemporal Evolution Heatmap
# ==============================================================================
evolution_matrix = np.vstack(compressed_snapshots)
step_array = np.arange(1, len(compressed_snapshots) + 1) * 4000

plt.figure(figsize=(14, 8))
limit = np.percentile(np.abs(evolution_matrix), 95)

pcm = plt.pcolormesh(
    z_array_compressed,
    step_array,
    evolution_matrix,
    cmap='RdBu_r',
    vmin=-limit,
    vmax=limit,
    shading='auto'
)

cbar = plt.colorbar(pcm, pad=0.02)
cbar.set_label('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Simulation Step', fontsize=12)
plt.title(f'Spatiotemporal Evolution of the Longitudinal Wakefield\nContinuous Au51+ Train Injection (0 to {final_step:,} Steps)', fontsize=14, fontweight='bold')
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.ylim(np.min(step_array), np.max(step_array))
plt.grid(True, linestyle='--', color='black', alpha=0.2)
plt.tight_layout()
plt.show()


# ==============================================================================
# PRINT FINAL METRICS
# ==============================================================================
print("\n==========================================================")
print("   0.2c Au51+ WAKEFIELD GRADIENT STATISTICS")
print("==========================================================")
print(f"Maximum Accelerating Gradient : {global_max_ez:8.2f} MV/m")
print(f"Maximum Decelerating Gradient : {global_min_ez:8.2f} MV/m")
print(f"Average Absolute Gradient     : {global_avg_abs_ez:8.2f} MV/m")
print("==========================================================\n")

print("==========================================================")
print("   MACROSCOPIC AMBIPOLAR TOWING FIELD (DC FILTERED)")
print("==========================================================")
print(f"Max DC Accelerating Gradient  : {np.max(Ez_dc_filtered):8.2f} MV/m")
print(f"Max DC Decelerating Gradient  : {np.min(Ez_dc_filtered):8.2f} MV/m")
print(f"Average Net Forward Pressure  : {np.mean(Ez_dc_filtered):8.2f} MV/m")
print("==========================================================")

In [ ]:
# ==============================================================================
# CELL 5: LONGITUDINAL PHASE SPACE EXTRACTION (LITHIUM BASELINE)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pywarpx.particle_containers import ParticleContainerWrapper

print("Extracting Lithium ion arrays from active AMReX memory...")

# Target the lithium ions
li_pc = ParticleContainerWrapper('lithium_ions')
z_li = np.concatenate(li_pc.get_particle_z(copy_to_host=True))
uz_li = np.concatenate(li_pc.get_particle_uz(copy_to_host=True))

stride = 10
z_plot = z_li[::stride]
vz_plot = uz_li[::stride]

plt.figure(figsize=(12, 6))
plt.scatter(z_plot * 1e3, vz_plot, s=4, alpha=0.7, color='green', edgecolors='none')

plt.axhline(0, color='black', linewidth=1, linestyle='-')
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Longitudinal Velocity, $v_z$ (m/s)', fontsize=12)
plt.title('Lithium Ion Phase Space: Macroscopic Forward Towing\nat Step 120,000', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()